# UDMS Image Classifier — Google Colab Training

**MobileNetV2 transfer learning · Two-phase training · TFLite export**

> ⚠️ **Before running:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`

| Step | Cell | Description |
|------|------|-------------|
| 1 | Mount Drive | Mount Google Drive — dataset already lives there |
| 2 | GitHub sync | Clone / pull latest code from GitHub |
| 3 | Install & GPU | Install extra deps, verify T4 GPU |
| 4 | Config | All constants in one place — edit here only |
| 5 | Data | Load splits, class chart, batch preview |
| 6 | Build model | Inline MobileNetV2 + Lambda preprocessing |
| 7 | Phase 1 | Train Dense head (backbone frozen) |
| 8 | Phase 2 | Fine-tune top-30 backbone layers |
| 9 | Evaluate | Test accuracy, report, confusion matrix |
| 10 | Export | TFLite (quantised) + smoke-test + benchmark |
| 11 | Save | Artifacts are already on Drive — or download to browser |

**Workflow:**
```
VS Code (edit + Copilot)
        ↓ git push
GitHub  (source of truth)
        ↓ git clone / pull  ← Step 2 below
Colab   (train on T4 GPU)
```

**Key design decision:** raw `float32` pixels `[0, 255]` flow through the whole pipeline.  
A `Lambda` layer inside the model applies `mobilenet_v2.preprocess_input` → `[-1, 1]`.  
This is baked into the exported `.tflite` — the API never needs to scale pixels manually.


---
## 1 · Mount Google Drive

Your dataset must already be at:
```
MyDrive/udms-project/data/processed/all/
  bad_drainage/
  damaged_signage/
  illegal_dumping/
  potholes/
  vegetation_overgrowth/
```
Trained models will be saved to `MyDrive/udms-project/models/` automatically.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---
## 2 · Clone / pull repo from GitHub

Edit `GITHUB_REPO` below to match your username/repo.  
Every time you push from VS Code, re-run this cell in Colab to pull the latest code.

**One-time setup** — authorize Colab to read private repos (if needed):
```python
from google.colab import userdata   # store your token in Colab Secrets as GH_TOKEN
```
For a public repo no token is required.


In [ ]:
import os, subprocess, sys

GITHUB_USERNAME = 'Light-Brain237'
GITHUB_REPO     = 'udms-image-classifier'
BRANCH          = 'main'

REPO_PATH = f'/content/{GITHUB_REPO}'
REPO_URL  = f'https://github.com/{GITHUB_USERNAME}/{GITHUB_REPO}.git'

if os.path.isdir(os.path.join(REPO_PATH, '.git')):
    print(f'Pulling latest changes into {REPO_PATH} ...')
    result = subprocess.run(
        ['git', '-C', REPO_PATH, 'pull', 'origin', BRANCH],
        capture_output=True, text=True,
    )
    print(result.stdout or result.stderr)
else:
    print(f'Cloning {GITHUB_USERNAME}/{GITHUB_REPO} ...')
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_PATH], check=True)
    print('Repository cloned.')

if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)
os.chdir(REPO_PATH)
print(f'Working directory: {os.getcwd()}')

result = subprocess.run(
    ['git', '-C', REPO_PATH, 'log', '--oneline', '-3'],
    capture_output=True, text=True,
)
print(f'Recent commits:\n{result.stdout}')


In [ ]:
# Verify the flat dataset directory exists on Drive
import os

def check_flat_dataset(data_dir):
    if not os.path.isdir(data_dir):
        print(f'NOT FOUND: {data_dir}')
        print('Top-level MyDrive contents:')
        for item in sorted(os.listdir('/content/drive/MyDrive')):
            print(f'  {item}')
        return
    classes = sorted([
        d for d in os.listdir(data_dir)
        if os.path.isdir(os.path.join(data_dir, d))
    ])
    print(f'Dataset found at: {data_dir}')
    print(f'Classes ({len(classes)}): {classes}')
    for cls in classes:
        count = len(os.listdir(os.path.join(data_dir, cls)))
        print(f'  {cls:<25} {count:>5,} images')

check_flat_dataset('/content/drive/MyDrive/udms-project/data/processed/all')


---
## 2 · Install extra dependencies & verify GPU


In [ ]:
import subprocess, sys

# Colab ships TensorFlow, NumPy, Matplotlib — install only what's missing
pkgs = ['scikit-learn>=1.3', 'pillow>=10.0', 'albumentations>=1.3']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs)

import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if not gpus:
    raise RuntimeError(
        'No GPU detected!\n'
        'Go to Runtime → Change runtime type → T4 GPU and re-run all cells.'
    )

print(f'TensorFlow : {tf.__version__}')
print(f'GPU        : {gpus[0].name}')
os.system('nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader')
print('All dependencies ready.')


---
## 3 · Configuration & constants

Edit values here — all other cells read from these variables.


In [ ]:
import os, json, time, shutil
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Dataset location (flat — one subfolder per class, no train/val/test)
DATA_DIR   = '/content/drive/MyDrive/udms-project/data/processed/all'
MODELS_DIR = '/content/drive/MyDrive/udms-project/models'

# Weights-only checkpoints — avoids Lambda/focal_loss serialization entirely
PHASE1_CKPT   = f'{MODELS_DIR}/phase1_best.weights.h5'
PHASE2_CKPT   = f'{MODELS_DIR}/phase2_best.weights.h5'
TFLITE_OUT    = f'{MODELS_DIR}/classifier.tflite'
LABEL_MAP_OUT = f'{MODELS_DIR}/label_map.json'

IMG_SIZE        = (224, 224)
INPUT_SHAPE     = (224, 224, 3)
BATCH_SIZE      = 32
NUM_CLASSES     = 5
TEMP_SPLIT      = 0.30
SEED            = 42
PHASE1_LR       = 1e-3
PHASE1_EPOCHS   = 20
PHASE2_LR       = 1e-5
PHASE2_EPOCHS   = 15
UNFREEZE_LAYERS = 30
EARLY_STOP_PAT  = 5

UDMS_CATEGORIES = [
    'bad_drainage', 'damaged_signage', 'illegal_dumping',
    'potholes', 'vegetation_overgrowth',
]
CATEGORY_LABELS = {
    'bad_drainage':          'Bad Drainage / Water Sewage Issues',
    'damaged_signage':       'Damaged Signage / Infrastructure',
    'illegal_dumping':       'Illegal Dumping / Garbage',
    'potholes':              'Pothole / Road Damage',
    'vegetation_overgrowth': 'Vegetation Overgrowth',
}

os.makedirs(MODELS_DIR, exist_ok=True)

# Delete stale .keras checkpoints from Drive — they contain the old Lambda
# layer and can never be loaded.  The new .weights.h5 files are weights-only.
for stale in ['phase1_best.keras', 'phase2_best.keras']:
    stale_path = os.path.join(MODELS_DIR, stale)
    if os.path.exists(stale_path):
        os.remove(stale_path)
        print(f'Removed stale checkpoint: {stale}')

if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(
        f'Dataset not found: {DATA_DIR}\n'
        'Upload data/processed/all/ to MyDrive/udms-project/data/processed/'
    )

classes_found = sorted([
    d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, d))
])
print(f'Data dir    : {DATA_DIR}')
print(f'Models dir  : {MODELS_DIR}')
print(f'Classes ({len(classes_found)}): {classes_found}')
for cls in classes_found:
    count = len(os.listdir(os.path.join(DATA_DIR, cls)))
    print(f'  {cls:<25} {count:>5,} images')
print('Configuration loaded.')


---
## 4 · Load data, inspect class distribution & preview a batch

> Images are loaded as raw `float32` in **[0, 255]**.  
> The MobileNetV2 `[-1, 1]` scaling is applied *inside the model* — never divide by 255 here.


In [ ]:
import os, shutil

# Always wipe the cache dir so stale lockfiles never block a new run.
CACHE_DIR = '/content/tf_cache'
if os.path.exists(CACHE_DIR):
    shutil.rmtree(CACHE_DIR)
os.makedirs(CACHE_DIR, exist_ok=True)
print(f'tf.data cache dir cleared and ready: {CACHE_DIR}')

# Scan and remove corrupt/unreadable image files before training.
from PIL import Image as _PIL_Image
import pathlib as _pl

_bad_files = []
for _p in _pl.Path(DATA_DIR).rglob('*'):
    if not _p.is_file():
        continue
    try:
        with _PIL_Image.open(_p) as _im:
            _im.verify()
    except Exception:
        _bad_files.append(_p)

if _bad_files:
    print(f'Removing {len(_bad_files)} corrupt/unreadable files...')
    for _p in _bad_files:
        _p.unlink()
        print(f'  removed: {_p.name}')
else:
    print('No corrupt files found — dataset is clean.')


In [ ]:

# ── Augmentation layers (training only, on GPU inside tf.data) ──────────────
_augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal_and_vertical'),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomBrightness(0.25),
    tf.keras.layers.RandomContrast(0.25),
    tf.keras.layers.RandomTranslation(0.1, 0.1),
], name='augmentation')


def _base_ds(subset: str) -> tf.data.Dataset:
    """Load a subset (unbatched) from flat DATA_DIR using TEMP_SPLIT=0.30."""
    ds = tf.keras.utils.image_dataset_from_directory(
        DATA_DIR,
        validation_split=TEMP_SPLIT,
        subset=subset,
        seed=SEED,
        image_size=IMG_SIZE,
        batch_size=None,
        label_mode='categorical',
    )
    return ds.map(
        lambda x, y: (tf.cast(x, tf.float32), y),
        num_parallel_calls=tf.data.AUTOTUNE,
    )


# ── Training split (70%) ──────────────────────────────────────────────────────
train_raw = _base_ds('training')
train_raw = train_raw.cache(os.path.join(CACHE_DIR, 'train'))
train_raw = train_raw.shuffle(4096, seed=SEED)
train_raw = train_raw.map(
    lambda x, y: (_augment(x, training=True), y),
    num_parallel_calls=tf.data.AUTOTUNE,
)
train_ds = train_raw.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# ── Temp split (30%) — split 50/50 into val (15%) and test (15%) ──────────────
temp_raw = _base_ds('validation')
temp_raw = temp_raw.cache(os.path.join(CACHE_DIR, 'temp'))
# Warm the cache before counting (required for tf.data cache to be seekable)
n_temp = sum(1 for _ in temp_raw)
n_val  = n_temp // 2

val_ds  = temp_raw.take(n_val).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds = temp_raw.skip(n_val).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# CLASS_NAMES is alphabetical — matches index order from image_dataset_from_directory.
CLASS_NAMES = sorted([
    d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, d))
])
print(f'Class names (index 0-{NUM_CLASSES-1}): {CLASS_NAMES}')


def count_samples(ds: tf.data.Dataset) -> int:
    return sum(int(x.shape[0]) for x, _ in ds)


n_train = count_samples(train_ds)
n_val_c = count_samples(val_ds)
n_test  = count_samples(test_ds)
total   = n_train + n_val_c + n_test

print(f'Train : {n_train:>5,} images  (~70%  → actual {n_train/total:.1%})')
print(f'Val   : {n_val_c:>5,} images  (~15%  → actual {n_val_c/total:.1%})')
print(f'Test  : {n_test:>5,} images  (~15%  → actual {n_test/total:.1%})')
print(f'Total : {total:>5,} images')
print(f'Batch shape: {next(iter(train_ds))[0].shape}')


In [ ]:
# Per-class sample counts from flat DATA_DIR
CLASS_NAMES = sorted([
    d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, d))
])

class_counts = np.array([
    len(os.listdir(os.path.join(DATA_DIR, cls)))
    for cls in CLASS_NAMES
])

fig, ax = plt.subplots(figsize=(10, 4))
colors = plt.cm.tab10(np.linspace(0, 0.9, NUM_CLASSES))
bars = ax.barh(CLASS_NAMES, class_counts, color=colors)
ax.bar_label(bars, padding=4, fontsize=9)
ax.set_xlabel('Total images')
ax.set_title('Dataset class distribution (all images before split)')
ax.set_xlim(0, class_counts.max() * 1.15)
plt.tight_layout()
plt.show()

for cls, cnt in zip(CLASS_NAMES, class_counts):
    print(f'  {cls:<25} {cnt:>5,}')
print(f'  {"TOTAL":<25} {class_counts.sum():>5,}')


In [ ]:
# Class weights to counteract imbalance
_counts  = np.array([
    len(os.listdir(os.path.join(DATA_DIR, cls)))
    for cls in CLASS_NAMES
])
_total   = _counts.sum()
_weights = _total / (NUM_CLASSES * _counts)

class_weights = {i: float(w) for i, w in enumerate(_weights)}

print('Class weights (balanced):')
print(f'  {"Index":<6} {"Folder":<25} {"Images":>8}  {"Weight":>8}')
print('  ' + '-' * 55)
for i, (cls, cnt, w) in enumerate(zip(CLASS_NAMES, _counts, _weights)):
    bar = chr(9608) * min(int(w * 2), 30)
    print(f'  [{i}]   {cls:<25} {cnt:>8,}  {w:>8.3f}  {bar}')
print(f'\n  Total images  : {_total:,}')
print(f'  Max weight    : {_weights.max():.3f}  ({CLASS_NAMES[int(_weights.argmax())]})')
print(f'  Min weight    : {_weights.min():.3f}  ({CLASS_NAMES[int(_weights.argmin())]})')


In [ ]:

# ── Focal loss ────────────────────────────────────────────────────────────────
# Standard categorical cross-entropy treats every example equally.
# Focal loss down-weights the easy, well-classified examples so the model
# focuses gradient updates on the hard / rarely-seen cases.
#
#   FL(p_t) = -(1 - p_t)^gamma * log(p_t)
#
# gamma=2.0 is the default from the original paper (Lin et al., 2017).
# Used together with class_weight= in model.fit() for a combined effect.

def focal_loss(gamma: float = 2.0):
    """Return a focal-loss function for multi-class classification.

    Args:
        gamma: Focusing parameter.  0 → standard cross-entropy; 2 is typical.
    """
    def loss_fn(y_true, y_pred):
        y_pred  = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        ce      = tf.reduce_sum(-y_true * tf.math.log(y_pred), axis=-1)
        p_t     = tf.reduce_sum(y_true * y_pred, axis=-1)          # prob of true class
        focal_w = tf.pow(1.0 - p_t, gamma)
        return tf.reduce_mean(focal_w * ce)

    loss_fn.__name__ = f'focal_loss_gamma{gamma}'
    return loss_fn


print(f'Focal loss ready  (gamma=2.0)')
print('Phase 1 & 2 will use focal_loss instead of categorical_crossentropy.')


---
## 5 · Build MobileNetV2 model

```
Input  (224×224×3, float32 [0, 255])
  └─ Lambda: mobilenet_v2.preprocess_input   →  [-1, 1]
  └─ MobileNetV2 backbone (ImageNet weights, frozen in Phase 1)
  └─ GlobalAveragePooling2D
  └─ Dropout(0.3)
  └─ Dense(128, relu)
  └─ Dropout(0.2)
  └─ Dense(5, softmax)
```


In [ ]:
def build_model(freeze_backbone: bool = True) -> tf.keras.Model:
    """Build MobileNetV2 transfer-learning classifier.

    Preprocessing (x/127.5 - 1) is applied via a Rescaling layer baked into
    the model — fully serializable, no Lambda / custom_objects needed.
    """
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=INPUT_SHAPE,
        include_top=False,
        weights='imagenet',
    )
    base_model.trainable = not freeze_backbone

    inputs = tf.keras.Input(shape=INPUT_SHAPE, name='image_input')

    # MobileNetV2 expects [-1, 1].  Rescaling(1/127.5, -1) is identical to
    # mobilenet_v2.preprocess_input and is natively serializable by Keras 3.
    x = tf.keras.layers.Rescaling(scale=1.0/127.5, offset=-1.0,
                                   name='mobilenetv2_preprocess')(inputs)
    x = base_model(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D(name='gap')(x)
    x = tf.keras.layers.Dropout(0.3, name='dropout_1')(x)
    x = tf.keras.layers.Dense(128, activation='relu', name='dense_128')(x)
    x = tf.keras.layers.Dropout(0.2, name='dropout_2')(x)
    outputs = tf.keras.layers.Dense(
        NUM_CLASSES, activation='softmax', name='predictions'
    )(x)

    return tf.keras.Model(inputs, outputs, name='udms_mobilenetv2')


model = build_model(freeze_backbone=True)
model.summary(line_length=90)

trainable     = sum(tf.size(w).numpy() for w in model.trainable_weights)
non_trainable = sum(tf.size(w).numpy() for w in model.non_trainable_weights)
print(f'\nTrainable params     : {trainable:,}')
print(f'Non-trainable params : {non_trainable:,}')


In [ ]:

def plot_history(history, title_prefix: str):
    epochs = range(1, len(history.history['accuracy']) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
    ax1.plot(epochs, history.history['accuracy'],     'b-o', ms=4, label='train')
    ax1.plot(epochs, history.history['val_accuracy'], 'r-o', ms=4, label='val')
    ax1.set_title(f'{title_prefix} — Accuracy')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
    ax1.legend(); ax1.grid(True, alpha=0.3)
    ax2.plot(epochs, history.history['loss'],     'b-o', ms=4, label='train')
    ax2.plot(epochs, history.history['val_loss'], 'r-o', ms=4, label='val')
    ax2.set_title(f'{title_prefix} — Loss')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
    ax2.legend(); ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=PHASE1_LR),
    loss=focal_loss(gamma=2.0),
    metrics=['accuracy'],
)

phase1_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=PHASE1_CKPT,
        monitor='val_accuracy',
        save_best_only=True,
        save_weights_only=True,   # weights only — no architecture serialization
        mode='max',
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=EARLY_STOP_PAT,
        restore_best_weights=True,
        verbose=1,
    ),
    tf.keras.callbacks.CSVLogger(f'{MODELS_DIR}/phase1_log.csv'),
]

print('=' * 60)
print('PHASE 1 — Training classification head (backbone frozen)')
print('=' * 60)

history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=PHASE1_EPOCHS,
    callbacks=phase1_callbacks,
    class_weight=class_weights,
)

print(f'\nPhase 1 complete — best val_accuracy: {max(history1.history["val_accuracy"]):.4f}')
plot_history(history1, 'Phase 1')


---
## 6 · Phase 1 — Train classification head

MobileNetV2 backbone is **completely frozen** — only the Dense head learns.

| Setting | Value |
|---------|-------|
| Optimizer | Adam lr=1e-3 |
| Max epochs | 20 |
| EarlyStopping | patience=5 on `val_loss` |
| Checkpoint | `models/phase1_best.keras` |


In [ ]:

backbone = next(
    l for l in model.layers
    if 'mobilenet' in l.name.lower() and hasattr(l, 'layers')
)
backbone.trainable = True
for layer in backbone.layers[:-UNFREEZE_LAYERS]:
    layer.trainable = False

trainable_now = sum(tf.size(w).numpy() for w in model.trainable_weights)
print(f'Trainable params after unfreeze : {trainable_now:,}')
print(f'Unfrozen backbone layers        : {UNFREEZE_LAYERS}')

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=PHASE2_LR),
    loss=focal_loss(gamma=2.0),
    metrics=['accuracy'],
)

phase2_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=PHASE2_CKPT,
        monitor='val_accuracy',
        save_best_only=True,
        save_weights_only=True,   # weights only — no architecture serialization
        mode='max',
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=EARLY_STOP_PAT,
        restore_best_weights=True,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1,
    ),
    tf.keras.callbacks.CSVLogger(f'{MODELS_DIR}/phase2_log.csv'),
]

print('\n' + '='*60)
print('PHASE 2 — Fine-tuning top-30 backbone layers')
print('='*60)

history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=PHASE2_EPOCHS,
    callbacks=phase2_callbacks,
    class_weight=class_weights,
)

best_p2 = max(history2.history['val_accuracy'])
print(f'\nPhase 2 complete — best val_accuracy: {best_p2:.4f}')


In [ ]:
plot_history(history2, 'Phase 2')
print(f'Best val accuracy (Phase 2): {max(history2.history["val_accuracy"]):.4f}')

# ── Combined validation-accuracy curve across both phases ─────────────────────
acc_all = history1.history['val_accuracy'] + history2.history['val_accuracy']
split_at = len(history1.history['val_accuracy'])

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(1, split_at + 1),       acc_all[:split_at],  'b-o', ms=4, label='Phase 1')
ax.plot(range(split_at, len(acc_all) + 1), acc_all[split_at - 1:], 'r-o', ms=4, label='Phase 2')
ax.axvline(split_at + 0.5, color='orange', linestyle='--', linewidth=1.5, label='Phase 1 → 2')
ax.set_title('Validation accuracy — both phases combined')
ax.set_xlabel('Epoch (cumulative)')
ax.set_ylabel('Val accuracy')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'\nSummary:')
print(f'  Phase 1 best val_accuracy : {max(history1.history["val_accuracy"]):.4f}')
print(f'  Phase 2 best val_accuracy : {max(history2.history["val_accuracy"]):.4f}')


---
## 7 · Phase 2 — Fine-tune top-30 backbone layers

Top `UNFREEZE_LAYERS` layers of MobileNetV2 are unfrozen and trained at a very low LR
to avoid destroying ImageNet features.

| Setting | Value |
|---------|-------|
| Optimizer | Adam lr=1e-5 |
| Max epochs | 15 |
| EarlyStopping | patience=5 on `val_loss` |
| ReduceLROnPlateau | factor=0.5, patience=3, min_lr=1e-7 |
| Checkpoint | `models/phase2_best.keras` |


In [ ]:
# ── Rebuild model + load Phase-2 weights ──────────────────────────────────
# PHASE2_CKPT is a WEIGHTS-ONLY file (.weights.h5).
# We must rebuild the exact same architecture, then call load_weights().
# Never use load_model() on a weights-only checkpoint.

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

CLASS_NAMES = [
    'bad_drainage', 'damaged_signage', 'illegal_dumping',
    'potholes', 'vegetation_overgrowth',
]
NUM_CLASSES = 5
INPUT_SHAPE = (224, 224, 3)

def _rebuild_model() -> tf.keras.Model:
    """Rebuild exact architecture used during training.
    Rescaling(1/127.5, -1) is identical to mobilenet_v2.preprocess_input
    and is natively serializable — no Lambda, no custom_objects needed.
    """
    base = tf.keras.applications.MobileNetV2(
        input_shape=INPUT_SHAPE, include_top=False, weights='imagenet'
    )
    base.trainable = True  # match Phase-2 state so weight tensor shapes align

    inputs = tf.keras.Input(shape=INPUT_SHAPE, name='image_input')
    x = tf.keras.layers.Rescaling(
        scale=1.0/127.5, offset=-1.0, name='mobilenetv2_preprocess'
    )(inputs)
    x = base(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D(name='gap')(x)
    x = tf.keras.layers.Dropout(0.3, name='dropout_1')(x)
    x = tf.keras.layers.Dense(128, activation='relu', name='dense_128')(x)
    x = tf.keras.layers.Dropout(0.2, name='dropout_2')(x)
    outputs = tf.keras.layers.Dense(
        NUM_CLASSES, activation='softmax', name='predictions'
    )(x)
    return tf.keras.Model(inputs, outputs, name='udms_mobilenetv2')


best_model = _rebuild_model()
best_model.load_weights(PHASE2_CKPT)   # load_weights(), NOT load_model()
print(f'Weights loaded from: {PHASE2_CKPT}')

# ── Evaluate on test set ───────────────────────────────────────────────────
y_true_list, y_prob_list = [], []
for images, labels in test_ds:
    probs = best_model.predict(images, verbose=0)
    y_prob_list.append(probs)
    y_true_list.append(np.argmax(labels.numpy(), axis=1))  # one-hot -> int

y_true = np.concatenate(y_true_list)
y_prob = np.concatenate(y_prob_list)
y_pred = np.argmax(y_prob, axis=1)

test_acc = accuracy_score(y_true, y_pred)
print(f'Test accuracy : {test_acc:.4f}  ({test_acc * 100:.2f}%)')
print(f'Test samples  : {len(y_true):,}')
print()
print('Classification Report:')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3))

# ── Confusion matrix ──────────────────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax,
)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title(f'Confusion Matrix — Test Accuracy: {test_acc:.2%}')
plt.xticks(rotation=35, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:

# ── Confusion matrix ──────────────────────────────────────────────────────────
cm     = confusion_matrix(y_true, y_pred)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm_pct, cmap='Blues', vmin=0, vmax=100)
plt.colorbar(im, ax=ax, label='% of true class')

ticks = np.arange(NUM_CLASSES)
ax.set_xticks(ticks)
ax.set_yticks(ticks)
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(CLASS_NAMES, fontsize=9)

for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        color = 'white' if cm_pct[i, j] > 50 else 'black'
        ax.text(j, i, f'{cm[i, j]}\n({cm_pct[i, j]:.0f}%)',
                ha='center', va='center', fontsize=7, color=color)

ax.set_ylabel('True label')
ax.set_xlabel('Predicted label')
ax.set_title(f'Confusion Matrix — test set  (accuracy {test_acc:.4f})')
plt.tight_layout()
plt.show()


In [ ]:

# ── Per-class confidence distribution (correct vs wrong predictions) ──────────
fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharey=True)
for idx, (ax, cls) in enumerate(zip(axes.flat, CLASS_NAMES)):
    mask    = y_true == idx
    if mask.sum() == 0:
        ax.set_visible(False)
        continue
    correct   = y_prob[mask & (y_pred == y_true), idx]
    incorrect = y_prob[mask & (y_pred != y_true), idx]
    ax.hist(correct,   bins=20, range=(0, 1), alpha=0.7, color='steelblue', label=f'correct ({len(correct)})')
    ax.hist(incorrect, bins=20, range=(0, 1), alpha=0.7, color='tomato',    label=f'wrong ({len(incorrect)})')
    ax.set_title(cls, fontsize=8)
    ax.set_xlabel('Model confidence', fontsize=7)
    ax.legend(fontsize=6)

# Hide the unused 8th subplot
axes.flat[-1].set_visible(False)

plt.suptitle('Confidence distribution per class — blue=correct, red=misclassified', fontsize=10)
plt.tight_layout()
plt.show()

# ── Top confused class pairs ──────────────────────────────────────────────────
print('Top misclassification pairs (true → predicted):')
off_diag = [(cm[i, j], CLASS_NAMES[i], CLASS_NAMES[j])
            for i in range(NUM_CLASSES) for j in range(NUM_CLASSES) if i != j]
for count, true_cls, pred_cls in sorted(off_diag, reverse=True)[:5]:
    print(f'  {true_cls:<22} → {pred_cls:<22}  ({count} samples)')


---
## 8 · Evaluate on test set


In [ ]:

# ── Convert to TFLite with dynamic-range quantisation ─────────────────────────
print('Converting to TFLite ...')
converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_bytes = converter.convert()

Path(TFLITE_OUT).write_bytes(tflite_bytes)
tflite_mb = Path(TFLITE_OUT).stat().st_size / 1024 / 1024
print(f'TFLite saved  → {TFLITE_OUT}  ({tflite_mb:.2f} MB)')

# ── Write label_map.json ──────────────────────────────────────────────────────
# IMPORTANT: use CLASS_NAMES (alphabetical order) — this matches the index that
# image_dataset_from_directory assigned during training. UDMS_CATEGORIES has a
# different order and would produce wrong predictions in the API.
label_map = {
    str(i): {'category': cls, 'label': CATEGORY_LABELS.get(cls, cls)}
    for i, cls in enumerate(CLASS_NAMES)
}
Path(LABEL_MAP_OUT).write_text(json.dumps(label_map, indent=2), encoding='utf-8')
print(f'Label map saved → {LABEL_MAP_OUT}')
print(json.dumps(label_map, indent=2))


In [ ]:

# ── Load TFLite model ─────────────────────────────────────────────────────────
interpreter = tf.lite.Interpreter(model_path=str(TFLITE_OUT))
interpreter.allocate_tensors()

inp_detail = interpreter.get_input_details()[0]
out_detail = interpreter.get_output_details()[0]

print('TFLite tensor details:')
print(f'  Input  shape={inp_detail["shape"]}  dtype={inp_detail["dtype"]}')
print(f'  Output shape={out_detail["shape"]}  dtype={out_detail["dtype"]}')

# ── Smoke-test with a real test image ────────────────────────────────────────
sample_imgs, sample_lbls = next(iter(test_ds))
sample      = sample_imgs[:1].numpy().astype(np.float32)   # shape (1,224,224,3)
true_class  = CLASS_NAMES[int(np.argmax(sample_lbls[0]))]

interpreter.set_tensor(inp_detail['index'], sample)
interpreter.invoke()
probs      = interpreter.get_tensor(out_detail['index'])[0]
pred_idx   = int(np.argmax(probs))
pred_conf  = float(probs[pred_idx])
pred_class = CLASS_NAMES[pred_idx]

print(f'\nSmoke-test result:')
print(f'  True  class : {true_class}')
print(f'  Pred  class : {pred_class}  (conf {pred_conf:.4f})')
print(f'  Probs sum   : {probs.sum():.6f}  (should be ~1.0)')
print(f'  Match       : {true_class == pred_class}')

# ── Inference benchmark (50 runs, CPU) ───────────────────────────────────────
BENCH_RUNS = 50
for _ in range(3):   # warm-up
    interpreter.set_tensor(inp_detail['index'], sample)
    interpreter.invoke()

times = []
for _ in range(BENCH_RUNS):
    t0 = time.perf_counter()
    interpreter.set_tensor(inp_detail['index'], sample)
    interpreter.invoke()
    _ = interpreter.get_tensor(out_detail['index'])
    times.append((time.perf_counter() - t0) * 1000)

print(f'\nInference benchmark ({BENCH_RUNS} runs, CPU):')
print(f'  Mean   : {np.mean(times):.1f} ms')
print(f'  Median : {np.median(times):.1f} ms')
print(f'  P95    : {np.percentile(times, 95):.1f} ms')
target_ok = np.mean(times) < 500
print(f'  Target <500 ms : {"PASS ✓" if target_ok else "FAIL ✗"}')


---
## 9 · Export to TFLite & smoke-test

Applies **dynamic-range quantisation** (INT8 weights, float32 I/O).  
The `mobilenetv2_preprocess` Lambda layer is baked into the artifact — callers send
raw `[0, 255]` pixels; the model handles the `[-1, 1]` scaling internally.


In [ ]:
# All artifacts were saved directly to Drive during training.
# List them here to confirm:
print(f'Artifacts in {MODELS_DIR}:\n')
for fname in sorted(os.listdir(MODELS_DIR)):
    fpath = os.path.join(MODELS_DIR, fname)
    size_kb = os.path.getsize(fpath) / 1024
    print(f'  {fname:<40} {size_kb:>8,.0f} KB')

print(f'\nTo deploy locally, copy classifier.tflite and label_map.json')
print(f'into your repo\'s models/ folder, then run:')
print(f'  uvicorn app.main:app --reload')

In [ ]:
# ── Option B: direct browser download ─────────────────────────────────────────
# Run this cell to download classifier.tflite and label_map.json directly.
# These are the only two files the API needs.
from google.colab import files

for fpath in [TFLITE_OUT, LABEL_MAP_OUT]:
    if os.path.exists(fpath):
        files.download(fpath)
        print(f'Downloading: {fpath}')
    else:
        print(f'Not found (skipped): {fpath}')


---
## Deploying the retrained model

All artifacts were saved to Drive during training — nothing was lost when the session ends:

```
MyDrive/udms-project/models/
  classifier.tflite    ← copy to your local  models/  folder
  label_map.json       ← copy to your local  models/  folder
  phase2_best.keras    ← full Keras model for further fine-tuning
  phase1_best.keras
  phase1_log.csv
  phase2_log.csv
```

Copy `classifier.tflite` and `label_map.json` into your repo's `models/` folder, then:

```bash
uvicorn app.main:app --reload
```

---
### Artifacts summary

| File | Description |
|------|-------------|
| `classifier.tflite` | Quantised TFLite model (INT8 weights, float32 I/O) |
| `label_map.json` | Index → category mapping used by the API |
| `phase2_best.keras` | Full Keras model — use for further fine-tuning |
| `phase1_log.csv` | Per-epoch metrics for Phase 1 |
| `phase2_log.csv` | Per-epoch metrics for Phase 2 |

In [ ]:
import os, subprocess

# ── Push any changes made in Colab back to GitHub ─────────────────────────────
# Run this cell after training to commit artifacts (model weights, logs, etc.)
# back to the repository.
# REPO_PATH is defined in the GitHub sync cell above — re-run that cell first
# if this cell is run in a fresh session.
# ─────────────────────────────────────────────────────────────────────────────

# Configure git identity
subprocess.run(["git", "-C", REPO_PATH, "config", "user.email", "kosianisiobi34@gmail.com"], check=True)
subprocess.run(["git", "-C", REPO_PATH, "config", "user.name",  "alexandrakosi"],            check=True)
print("Git identity configured.")

# Stage all changes
subprocess.run(["git", "-C", REPO_PATH, "add", "."], check=True)
print("All changes staged.")

# Commit only if there is something to commit
status = subprocess.run(
    ["git", "-C", REPO_PATH, "status", "--porcelain"],
    capture_output=True, text=True,
).stdout.strip()

if status:
    subprocess.run(
        ["git", "-C", REPO_PATH, "commit", "-m", "feat: update from Colab session"],
        check=True,
    )
    print("Changes committed: 'feat: update from Colab session'")
else:
    print("Nothing to commit — working tree is clean.")

# Push to origin main
subprocess.run(["git", "-C", REPO_PATH, "push", "origin", "main"], check=True)
print("Changes pushed to origin/main successfully.")
